<a href="https://colab.research.google.com/github/Nanda-Lopes/AlgoStudies/blob/main/Stanford_Algorithms_Specialization_Course3_W3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importações e Downloads

In [ ]:
import os
import urllib.request

if not os.path.exists("huffman.txt"):
    url = "https://raw.githubusercontent.com/bea-muller/stanford-algorithms/master/course3_greedy_algorithms_minimum_spanning_trees_and_dynamic_programming/week3_huffman_and_mwis/huffman.txt"
    urllib.request.urlretrieve(url, "huffman.txt")

if not os.path.exists("mwis.txt"):
    url = "https://raw.githubusercontent.com/bea-muller/stanford-algorithms/master/course3_greedy_algorithms_minimum_spanning_trees_and_dynamic_programming/week3_huffman_and_mwis/mwis.txt"
    urllib.request.urlretrieve(url, "mwis.txt")

print("Arquivos prontos para uso!")

Arquivos prontos para uso!


# Huffman
O algoritmo guloso de Huffman constrói um código de prefixo ótimo para um conjunto de caracteres com base em suas frequências ou pesos. O objetivo é minimizar o comprimento médio ponderado das palavras de código.

A estratégia de resolução consiste nas seguintes etapas:
1. Cada símbolo foi representado inicialmente como uma árvore de um único nó.
2. Todas as árvores foram inseridas em uma fila de prioridade (Min-Heap) ordenada por peso.
3. Em cada etapa, as duas árvores de menor peso foram removidas da fila e unidas sob uma nova raiz, cujo peso é a soma dos pesos dos dois filhos.
4. Esse processo foi repetido até restar apenas uma única árvore na fila, que representa a árvore de codificação final.
5. O comprimento de cada palavra de código corresponde exatamente à profundidade do respectivo nó folha na árvore final.

In [ ]:
import heapq

class HuffmanNode:
    def __init__(self, weight, symbol=None, left=None, right=None):
        self.weight = weight
        self.symbol = symbol
        self.left = left
        self.right = right

    def __lt__(self, other):
        return self.weight < other.weight

def solve_huffman(filename):
    with open(filename, "r") as f:
        lines = f.readlines()
    num_symbols = int(lines[0].strip())
    heap = []
    for i, line in enumerate(lines[1:]):
        if line.strip():
            w = int(line.strip())
            node = HuffmanNode(w, symbol=i)
            heapq.heappush(heap, node)
    while len(heap) > 1:
        n1 = heapq.heappop(heap)
        n2 = heapq.heappop(heap)
        merged = HuffmanNode(n1.weight + n2.weight, left=n1, right=n2)
        heapq.heappush(heap, merged)
    root = heap[0] # Corrected: access the node from the list
    lengths = []
    def traverse(node, current_depth):
        # Base case: if it's a leaf node
        if node.left is None and node.right is None:
            lengths.append(current_depth)
            return
        # Recursive case: traverse children
        if node.left:
            traverse(node.left, current_depth + 1)
        if node.right:
            traverse(node.right, current_depth + 1)
    traverse(root, 0)
    return min(lengths), max(lengths)

min_len, max_len = solve_huffman("huffman.txt")
print("==================================")
print("RESPOSTA QUESTÃO 1 (MAX LENGTH):")
print(max_len)
print("RESPOSTA QUESTÃO 2 (MIN LENGTH):")
print(min_len)
print("==================================")

RESPOSTA QUESTÃO 1 (MAX LENGTH):
19
RESPOSTA QUESTÃO 2 (MIN LENGTH):
9


# MWIS
Conjunto Independente de Peso Máximo em Grafos de Caminho (MWIS)

O problema de encontrar o Conjunto Independente de Peso Máximo (MWIS) em um grafo de caminho de $n$ vértices é resolvido com Programação Dinâmica de forma linear. Vértices adjacentes no caminho não podem fazer parte do conjunto simultaneamente.

A abordagem matemática consiste em:

1. **Equação de Recorrência (Cálculo dos Valores):**
   Seja $A[i]$ o valor do conjunto independente de peso máximo para os primeiros $i$ vértices.
   - Casos base: $A = 0$ e $A[1] = w_1$.
   - Para cada vértice de $2$ até $n$: $A[i] = \max(A[i-1], A[i-2] + w_i)$.

2. **Procedimento de Reconstrução:**
   Após preencher o vetor de decisões, o algoritmo reconstrói o conjunto ótimo de trás para frente, de $n$ até $1$:
   - Se $A[i-1] \ge A[i-2] + w_i$, o vértice $i$ não pertence ao conjunto ótimo. O algoritmo passa a avaliar o vértice $i-1$.
   - Caso contrário, o vértice $i$ pertence ao conjunto. Ele foi adicionado à solução e o algoritmo passa a avaliar o vértice $i-2$.

No final, os vértices de interesse (1, 2, 3, 4, 17, 117, 517 e 997) foram verificados e incluídos na solução e geramos a cadeia binária de 8 bits correspondente.

In [ ]:
def solve_mwis(filename):
    with open(filename, "r") as f:
        lines = f.readlines()
    num_vertices = int(lines[0].strip()) # Fix 1: access first line
    weights = []
    for line in lines[1:]:
        if line.strip():
            weights.append(int(line.strip()))

    # Initialize DP array 'a' with 0s. a[0] = 0, a[i] stores max weight up to vertex i
    a = [0] * (num_vertices + 1) # Fix 2: SyntaxError and initialization

    # Base case for DP
    if num_vertices >= 1:
        a[1] = weights[0] # Fix 3: Assign first weight (weights is 0-indexed, a is 1-indexed)

    for i in range(2, num_vertices + 1):
        # Recurrence: max of (not including current vertex i) vs (including current vertex i)
        # weights[i-1] corresponds to the weight of vertex i
        a[i] = max(a[i-1], a[i-2] + weights[i-1])

    mwis_vertices = set()
    i = num_vertices
    while i >= 1:
        if i == 1:
            # If only one vertex left and it's chosen (a[1] was max)
            if a[1] > a[0]: # Check if vertex 1 was actually included
                mwis_vertices.add(1)
            i -= 1
        elif a[i-1] >= a[i-2] + weights[i-1]:
            # If including current vertex i does not yield a higher value, skip it
            i -= 1
        else:
            # If including current vertex i yields a higher value, add it
            mwis_vertices.add(i)
            i -= 2

    # Target vertices to check
    targets = [1, 2, 3, 4, 17, 117, 517, 997] # Fix 4: Correct target list
    result_bits = []
    for t in targets:
        if t in mwis_vertices:
            result_bits.append("1")
        else:
            result_bits.append("0")
    return "".join(result_bits)

mwis_binary_string = solve_mwis("mwis.txt")
print("==================================")
print("RESPOSTA QUESTÃO 3 (8-BIT STRING):")
print(mwis_binary_string)
print("==================================")

RESPOSTA QUESTÃO 3 (8-BIT STRING):
10100110
